# Sanctions Screening: Generate Dataset

Generates **10,000 holding–sanctioned entity pairs** for fuzzy-matching analysis.

| Component | Count |
|-----------|-------|
| Holdings (funds, properties, companies) | 100 |
| Sanctioned entities (Russian / international) | 100 |
| Cross-joined pairs | **10,000** |

Some sanctioned names intentionally mirror holding names (e.g. *Gazprom Energy Holdings* ↔ *Gazprom OOO*) to produce realistic true-positive matches alongside a majority of unrelated pairs.

In [ ]:
CATALOG = "renjiharold_demo"
SCHEMA = "sanctions_screening"
TABLE  = "holdings_sanctioned_pairs"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Using {CATALOG}.{SCHEMA}")

## Generate holdings and sanctioned entity lists

- **Holdings**: Combinatorial (base name × entity type) + hand-picked Russian-linked companies.
- **Sanctioned**: Combinatorial (transliterated base × legal suffix OOO/ZAO/AO) + hand-picked entities from the screenshot reference data.

In [ ]:
import itertools, random
random.seed(42)

# ── Holdings ──────────────────────────────────────────────────────────
holding_bases = [
    "Mizner", "Coronado", "Westroads", "Augusta", "Apache",
    "Southpoint", "Autumn", "Beltsville", "Telstra", "Horizon",
    "Spectra", "Northgate", "Riverdale", "Oakwood", "Silverstone",
    "Brookfield", "Pinnacle", "Summit", "Pacific", "Atlantic",
    "Meridian", "Crestview", "Lakewood", "Fairview", "Highland",
    "Grandview", "Bayshore", "Ironwood", "Cedarwood", "Maplewood",
    "Ridgewood", "Ashford", "Preston", "Windsor", "Kensington",
    "Berkeley", "Cambridge", "Oxford", "Hampton", "Burlington",
]

holding_types = [
    "Park", "Mall", "Center", "Towers", "Holdings",
    "Capital", "Properties", "Group", "Industries", "Vista",
    "Crossing", "Place", "Square", "Plaza", "Estates",
    "Trust", "REIT", "Logistics", "Office", "Automotive",
]

combos = [f"{b} {t}" for b, t in itertools.product(holding_bases, holding_types)]
random.shuffle(combos)
holdings = combos[:80]

holdings += [
    "L'Oreal Brazil Office", "BPR-FF JV LLC", "Center Parcs UK",
    "Simon Property Group", "Westfield Corp",
    "Gazprom Energy Holdings", "Rosneft Oil Capital", "Sberbank Financial",
    "Lukoil Petroleum", "Norilsk Mining Resources",
    "Transneft Pipeline Holdings", "Severstal Steel Industries",
    "Magnit Retail Group", "Yandex Technology Group", "Alfa Investment Capital",
    "Sistema Conglomerate", "Evraz Steel Industries", "Polyus Gold Mining",
    "Novatek Gas Energy", "Rusal Aluminum Corp",
]

# ── Sanctioned entities ───────────────────────────────────────────────
sanctioned_bases = [
    "Orizon", "Spektr", "Biznes-Park", "Centr", "Kolombina",
    "Siner", "Uilner", "Amital", "Tornado", "Logistik",
    "Spark", "Kugulta", "Komital", "Elstar", "Vikta",
    "Vinta", "Visla", "Mister", "Kazache", "Bentek",
    "Render", "Rentek", "Lestrans", "Vestros", "Kristall",
    "Granit", "Progress", "Energiya", "Vostok", "Zarya",
    "Molniya", "Strela", "Fakel", "Impuls", "Signal",
]

sanctioned_types = ["OOO", "ZAO", "AO"]

sanc_combos = [f"{b} {t}" for b, t in itertools.product(sanctioned_bases, sanctioned_types)]
random.shuffle(sanc_combos)
sanctioned = sanc_combos[:70]

sanctioned += [
    "Gazprom OOO", "Rosneft AO", "Sberbank ZAO", "Lukoil AO",
    "Norilsk Nikel OOO", "Transneft AO", "Surgutneftegaz OOO",
    "Severstal OOO", "Magnit OOO", "Yandeks OOO",
    "Alfa-Bank AO", "AFK Sistema OOO", "Evraz OOO",
    "Polyus Zoloto OOO", "Novatek OOO", "Rusal OOO",
    "L'Oreal ZAO", "SOUTHFRONT", "Kapital OOO",
    "Rostec AO", "Almaz-Antey OOO", "Kalashnikov ZAO",
    "UK Centr OOO", "xplace OOO", "LLC PPR",
    "ARK OOO", "Perk OOO", "PRK OOO", "Orion OOO",
    "Buratino OOO",
]

assert len(set(holdings)) == 100, f"Expected 100 unique holdings, got {len(set(holdings))}"
assert len(set(sanctioned)) == 100, f"Expected 100 unique sanctioned, got {len(set(sanctioned))}"
print(f"Holdings: {len(holdings)}  |  Sanctioned: {len(sanctioned)}  |  Expected pairs: {len(holdings) * len(sanctioned):,}")

## Cross-join and save to Delta table

In [ ]:
holdings_df   = spark.createDataFrame([(h,) for h in holdings],   ["holding"])
sanctioned_df = spark.createDataFrame([(s,) for s in sanctioned], ["sanctioned"])

pairs_df = holdings_df.crossJoin(sanctioned_df)

full_table = f"{CATALOG}.{SCHEMA}.{TABLE}"
pairs_df.write.mode("overwrite").saveAsTable(full_table)
print(f"Saved {pairs_df.count():,} pairs → {full_table}")

In [ ]:
display(spark.table(full_table).limit(20))